# Mammography augmentation visualization

Uses `core.transforms.ConfigurableMGAugmentation`, the same implementation as training. Edit the cluster data and augmentation-config paths below. Start Jupyter from the project root or `notebooks/`; upload `core/` alongside the notebook. The filename is retained for compatibility; the selected JSON determines the augmentation policy.


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Optional

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# Find the project whether Jupyter starts in src/ or src/notebooks/.
import sys

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "core" / "transforms.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from the project directory or its notebooks/ folder.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.config import deep_get
from core.data import MammographyDataset, ensure_original_index, prepare_labels, read_csv_clean
from core.data import _safe_json_loads as parse_context_maybe
from core.transforms import ConfigurableMGAugmentation

# -----------------------------
# Editable paths / parameters
# -----------------------------

FULL_CSV = Path("/pfss/mlde/workspaces/mlde_wsp_PI_Roig/shared/datasets/breastTumor/mg/mg-only-all.csv")
BIN_PATH = Path("/pfss/mlde/workspaces/mlde_wsp_PI_Roig/shared/datasets/breastTumor/mg/mg-only-all.bin")
AUG_CONFIG = Path("/pfss/mlde/workspaces/mlde_wsp_PI_Roig/luis/mg_lejepa_aug_v4.json")

# Prefer the debug train split so examples match the current training setting.
# Set SPLIT_CSV = FULL_CSV if you want to sample from all MG rows.
SPLIT_CSV = Path("/pfss/mlde/workspaces/mlde_wsp_PI_Roig/shared/datasets/breastTumor/mg/debug-split/mg_debug_30k_balanced_train.csv")

IMAGE_HEIGHT = 512
IMAGE_WIDTH = 512
MEMMAP_DTYPE = "uint16"
SEED = 42

# Display defaults. You can override these in later cells.
N_RANDOM_IMAGES = 4
N_AUGS_PER_IMAGE = 4

# For repeatable augmentation draws, call core.data.set_seed(SEED).

# -----------------------------
# Label / metadata helpers
# -----------------------------


def extract_view(context: Any) -> str:
    d = parse_context_maybe(context)
    view = d.get("exam", {}).get("view", None) if isinstance(d, dict) else None
    if view is None:
        return "unknown"
    return str(view).strip().lower()

def extract_laterality(context: Any) -> str:
    d = parse_context_maybe(context)
    lat = d.get("exam", {}).get("laterality", None) if isinstance(d, dict) else None
    if lat is None:
        return "unknown"
    return str(lat).strip().lower()

def machine_family(machine: Any) -> str:
    if pd.isna(machine):
        return "unknown"
    s = str(machine).strip()
    sl = s.lower()
    if not s or sl in {"nan", "none"}:
        return "unknown"
    if "hologic" in sl:
        return "hologic"
    if "ge" in sl or "senograph" in sl or "senographe" in sl:
        return "ge"
    if "howtek" in sl or "lumysis" in sl or "dba" in sl:
        return "howtek/lumysis"
    return "other"

def prepare_df(full_csv: Path, split_csv: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    full = read_csv_clean(full_csv)
    full["original_index"] = np.arange(len(full))
    split = ensure_original_index(read_csv_clean(split_csv), full, "visualization")
    split = prepare_labels(split)
    split["view"] = split["context"].apply(extract_view) if "context" in split.columns else "unknown"
    split["laterality"] = split["context"].apply(extract_laterality) if "context" in split.columns else "unknown"
    split["machine_family"] = split["machine"].apply(machine_family) if "machine" in split.columns else "unknown"
    return full, split.reset_index(drop=True)


# -----------------------------
# Config-driven augmentation
# -----------------------------


# -----------------------------
# Load data/config
# -----------------------------

with open(AUG_CONFIG, "r", encoding="utf-8") as f:
    aug_cfg = json.load(f)

IMAGE_SIZE = int(deep_get(aug_cfg, ["image", "output_size"], 224))
print(f"Using IMAGE_SIZE={IMAGE_SIZE} from {AUG_CONFIG}")

full_df, df = prepare_df(FULL_CSV, SPLIT_CSV)
images = MammographyDataset(
    full_df, BIN_PATH, len(full_df), (IMAGE_HEIGHT, IMAGE_WIDTH), MEMMAP_DTYPE,
    transform=torch.nn.Identity(),
)

train_transform = ConfigurableMGAugmentation(aug_cfg, IMAGE_SIZE, train=True)
eval_transform = ConfigurableMGAugmentation(aug_cfg, IMAGE_SIZE, train=False)

print(f"Loaded full rows: {len(full_df):,}")
print(f"Loaded split rows: {len(df):,}")
print("collapsed_birads:", df["collapsed_birads"].value_counts().to_dict())
print("view:", df["view"].value_counts().to_dict())
print("machine_family:", df["machine_family"].value_counts().to_dict())

print("Ready. Use the cells below to visualize augmentations.")

In [ ]:
# -----------------------------
# Visualization helpers
# -----------------------------

def load_tensor_from_row(row: pd.Series) -> torch.Tensor:
    return images.image_at(int(row["original_index"]))


def tensor_to_img(x: torch.Tensor) -> np.ndarray:
    return x.detach().cpu().squeeze(0).numpy().clip(0, 1)

def row_title(row: pd.Series) -> str:
    parts = [
        f"idx={int(row['original_index'])}",
        f"collapsed={row.get('collapsed_birads', '?')}",
        f"BI-RADS={row.get('birads_numeric', '?')}",
        f"view={row.get('view', '?')}",
        f"lat={row.get('laterality', '?')}",
        f"machine={row.get('machine_family', '?')}",
        f"dataset={row.get('dataset', '?')}",
    ]
    return " | ".join(parts)

def plot_rows_with_augmentations(
    rows: pd.DataFrame,
    n_augs: int = 4,
    title: str = "",
    categories: Optional[list[str]] = None,
):
    rows = rows.reset_index(drop=True)
    n_rows = len(rows)
    n_cols = 2 + int(n_augs)

    if categories is not None:
        categories = list(categories)
    
        if len(categories) == 0:
            raise ValueError("categories must not be empty")
    
        if len(categories) != n_rows:
            if n_rows % len(categories) == 0:
                repeat_factor = n_rows // len(categories)
                categories = [cat for cat in categories for _ in range(repeat_factor)]
            else:
                raise ValueError(f"categories must have length {n_rows}, got {len(categories)}. ")

    fig_w = max(3.0 * n_cols, 8)
    fig_h = max(3.2 * n_rows, 3.5)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(fig_w, fig_h),
        squeeze=False,
    )

    for r, (_, row) in enumerate(rows.iterrows()):
        x = load_tensor_from_row(row)
        original = x.clone()
        preprocessed = eval_transform(x.clone())

        images = [original, preprocessed] + [train_transform(x.clone()) for _ in range(n_augs)]
        labels = ["original", "preprocessed"] + [f"aug {i + 1}" for i in range(n_augs)]

        for c, (img, label) in enumerate(zip(images, labels)):
            ax = axes[r, c]
            ax.imshow(tensor_to_img(img), cmap="gray", vmin=0, vmax=1)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_frame_on(False)

            if r == 0:
                ax.set_title(label)

    # Only reserve left space if category labels are explicitly provided.
    left_margin = 0.08 if categories is None else 0.13
    top_margin = 0.92 if title else 0.97

    fig.subplots_adjust(
        left=left_margin,
        right=0.99,
        top=top_margin,
        bottom=0.03,
        wspace=0.03,
        hspace=0.12,
    )

    # Only draw left-side labels when categories is provided.
    if categories is not None:
        for r, label in enumerate(categories):
            first_ax = axes[r, 0]
            bbox = first_ax.get_position()
            y_center = (bbox.y0 + bbox.y1) / 2.0

            fig.text(
                left_margin - 0.035,
                y_center,
                str(label),
                ha="center",
                va="center",
                fontsize=12,
                fontweight="bold",
                rotation=90,  # bottom-to-top
            )

    if title:
        fig.suptitle(title, fontsize=14)

    plt.show()

def sample_random_rows(n: int, seed: int = SEED) -> pd.DataFrame:
    n = min(n, len(df))
    return df.sample(n=n, random_state=seed)

def sample_per_category(
    column: str,
    categories: Optional[list[str]] = None,
    nSamples: int = 1,
    seed: int | None = None,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)

    if nSamples < 1:
        raise ValueError(f"nSamples must be >= 1, got {nSamples}")

    if categories is None:
        categories = [
            c for c in sorted(df[column].dropna().unique().tolist())
            if str(c).lower() not in {"unknown", "nan", "none"}
        ]

    rows = []

    for cat in categories:
        sub = df[df[column].astype(str) == str(cat)]

        if len(sub) == 0:
            print(f"WARNING: no rows for {column}={cat}")
            continue

        # Sample without replacement if possible, otherwise with replacement.
        replace = len(sub) < nSamples
        sampled_indices = rng.choice(len(sub), size=nSamples, replace=replace)

        if replace:
            print(
                f"WARNING: only {len(sub)} rows for {column}={cat}; "
                f"sampling {nSamples} with replacement"
            )

        for idx in sampled_indices:
            rows.append(sub.iloc[int(idx)])

    if not rows:
        raise ValueError(f"No rows found for column={column}")

    return pd.DataFrame(rows).reset_index(drop=True)

In [ ]:
N_RANDOM_IMAGES = 12
N_AUGS_PER_IMAGE = 3

rows = sample_random_rows(N_RANDOM_IMAGES)
plot_rows_with_augmentations(rows, n_augs=N_AUGS_PER_IMAGE, title="Random MG examples with augmentations")

In [ ]:
rows = sample_per_category(
    "collapsed_birads",
    categories=["routine", "follow_up", "biopsy"]
)
plot_rows_with_augmentations(
    rows,
    n_augs=N_AUGS_PER_IMAGE,
    title="Examples per collapsed BI-RADS class",
    categories=["routine", "follow_up", "biopsy"],
)

In [ ]:
rows = sample_per_category(
    "view",
    categories=["cranial caudal", "mediolateral oblique"],
    nSamples=3
)
plot_rows_with_augmentations(
    rows,
    n_augs=N_AUGS_PER_IMAGE,
    title="Examples per mammography view",
    categories=["cranial caudal", "mediolateral oblique"],
)

In [ ]:
# Common families from the current MG metadata.
categories = ["hologic", "ge", "howtek/lumysis", "unknown", "other"]
categories = [c for c in categories if (df["machine_family"].astype(str) == c).any()]

rows = sample_per_category(
    "machine_family",
    categories=categories,
    seed=SEED
)
plot_rows_with_augmentations(
    rows,
    n_augs=N_AUGS_PER_IMAGE,
    title="Examples per machine family",
    categories=categories,
)

In [ ]:
dataset_categories = [
    "embed",
    "cbis-ddsm",
    "rsna-bcd",
    "tompei-cmmd",
    "inbreast",
    "breast-micro-calc",
]

# Keep only datasets that are actually present in the currently loaded split.
dataset_categories = [
    c for c in dataset_categories
    if (df["dataset"].astype(str) == c).any()
]

rows = sample_per_category(
    "dataset",
    categories=dataset_categories,
)

plot_rows_with_augmentations(
    rows,
    n_augs=N_AUGS_PER_IMAGE,
    title="Examples per dataset",
    categories=dataset_categories,
)